# Ablation Study: Max Pooling on QEvasion Dataset

This notebook trains on QEvasion train split and evaluates on QEvasion test split.
Uses **Max Pooling** with sliding window chunking (stride=256).

In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel, RobertaModel
from dataclasses import dataclass
from typing import Optional, Tuple, Any, Dict, List
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda'

## Configuration

In [ ]:
MODEL_NAME = "roberta-large"
MAX_LENGTH = 512
STRIDE = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

clarity_label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
clarity_id2label = {v: k for k, v in clarity_label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

evasion_classes = [
    'Declining to answer', 'Dodging', 'General', 'Explicit', 
    'Claims ignorance', 'Clarification', 'Implicit', 
    'Partial/half-answer', 'Deflection'
]
evasion_label2id = {label: idx for idx, label in enumerate(evasion_classes)}
evasion_id2label = {v: k for k, v in evasion_label2id.items()}

## Load QEvasion Dataset

In [ ]:
# Load dataset
print("Loading QEvasion dataset from HuggingFace...")
ds = load_dataset("ailsntua/QEvasion")

train_df = ds['train'].to_pandas()

print(f"Train samples: {len(train_df)}")

print("\nTrain clarity distribution:")
print(train_df['clarity_label'].value_counts())
print("\nTrain evasion distribution:")
print(train_df['evasion_label'].value_counts())

## Tokenization

In [ ]:
def tokenize_function(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=False,
        padding=False,
        max_length=None,
    )
    
    tokenized["clarity_labels"] = [clarity_label2id[l] for l in examples["clarity_label"]]
    tokenized["evasion_labels"] = [evasion_label2id[l] for l in examples["evasion_label"]]
    
    return tokenized

print("Tokenization function defined - will be applied per-fold during CV")

## Data Collator

In [ ]:
@dataclass
class MultiHeadDataCollator:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        clarity_labels = torch.tensor([f.pop("clarity_labels") for f in features], dtype=torch.long)
        evasion_labels = torch.tensor([f.pop("evasion_labels") for f in features], dtype=torch.long)
        
        batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        batch["clarity_labels"] = clarity_labels
        batch["evasion_labels"] = evasion_labels
        
        return batch

data_collator = MultiHeadDataCollator(tokenizer=tokenizer)

## Model Architecture: Max Pooling

In [ ]:
@dataclass
class MultiHeadClassifierOutput:
    loss: Optional[torch.FloatTensor] = None
    logits_clarity: torch.FloatTensor = None
    logits_evasion: torch.FloatTensor = None
    hidden_states: Optional[Tuple[torch.FloatTensor, ...]] = None
    attentions: Optional[Tuple[torch.FloatTensor, ...]] = None


class MultiHeadMaxPooling(RobertaPreTrainedModel):
    
    @classmethod
    def _can_set_experts_implementation(cls) -> bool:
        return False
    
    def __init__(self, config):
        super().__init__(config)
        self.num_clarity_labels = 3
        self.num_evasion_labels = 9
        self.config = config

        self.roberta = RobertaModel(config)

        classifier_dropout = (
            config.classifier_dropout 
            if hasattr(config, 'classifier_dropout') and config.classifier_dropout is not None 
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)

        self.classifier_clarity = nn.Linear(config.hidden_size, self.num_clarity_labels)
        self.classifier_evasion = nn.Linear(config.hidden_size, self.num_evasion_labels)

        self.post_init()
    
    def create_chunks_batched(self, input_ids, attention_mask, max_length=512, stride=256):
        batch_size, seq_len = input_ids.shape
        
        all_chunk_ids = []
        all_chunk_masks = []
        chunk_to_sample = [] 
        
        for sample_idx in range(batch_size):
            sample_input_ids = input_ids[sample_idx]
            sample_attention_mask = attention_mask[sample_idx]
            actual_length = sample_attention_mask.sum().item()

            if actual_length <= max_length:
                chunk_ids = sample_input_ids[:max_length]
                chunk_mask = sample_attention_mask[:max_length]
                
                if len(chunk_ids) < max_length:
                    padding_length = max_length - len(chunk_ids)
                    chunk_ids = torch.cat([
                        chunk_ids,
                        torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                    ])
                    chunk_mask = torch.cat([
                        chunk_mask,
                        torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)
                    ])
                
                all_chunk_ids.append(chunk_ids)
                all_chunk_masks.append(chunk_mask)
                chunk_to_sample.append(sample_idx)
            else:
                start = 0
                while start < actual_length:
                    end = min(start + max_length, actual_length)
                    
                    chunk_ids = sample_input_ids[start:end]
                    chunk_mask = sample_attention_mask[start:end]
                    
                    if len(chunk_ids) < max_length:
                        padding_length = max_length - len(chunk_ids)
                        chunk_ids = torch.cat([
                            chunk_ids,
                            torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                        ])
                        chunk_mask = torch.cat([
                            chunk_mask,
                            torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)
                        ])
                    
                    all_chunk_ids.append(chunk_ids)
                    all_chunk_masks.append(chunk_mask)
                    chunk_to_sample.append(sample_idx)
  
                    start += stride
                    if end >= actual_length:
                        break

        all_chunk_ids = torch.stack(all_chunk_ids, dim=0)
        all_chunk_masks = torch.stack(all_chunk_masks, dim=0)
        chunk_to_sample = torch.tensor(chunk_to_sample, dtype=torch.long, device=input_ids.device)
        
        return all_chunk_ids, all_chunk_masks, chunk_to_sample
    
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        clarity_labels=None,
        evasion_labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ) -> MultiHeadClassifierOutput:
        
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        batch_size = input_ids.shape[0]

        all_chunk_ids, all_chunk_masks, chunk_to_sample = self.create_chunks_batched(
            input_ids, attention_mask, max_length=512, stride=256
        )

        outputs = self.roberta(
            input_ids=all_chunk_ids,
            attention_mask=all_chunk_masks,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
        )

        all_cls_embeddings = outputs.last_hidden_state[:, 0, :]
        batch_embeddings = []
        for sample_idx in range(batch_size):
            sample_mask = chunk_to_sample == sample_idx
            sample_chunk_embeddings = all_cls_embeddings[sample_mask]
            
            # MAX POOLING
            pooled_embedding = torch.max(sample_chunk_embeddings, dim=0)[0]
            batch_embeddings.append(pooled_embedding)

        pooled_output = torch.stack(batch_embeddings, dim=0)
        pooled_output = self.dropout(pooled_output)

        logits_clarity = self.classifier_clarity(pooled_output)
        logits_evasion = self.classifier_evasion(pooled_output)

        loss = None
        if clarity_labels is not None and evasion_labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss_clarity = loss_fct(logits_clarity.view(-1, self.num_clarity_labels), clarity_labels.view(-1))
            loss_evasion = loss_fct(logits_evasion.view(-1, self.num_evasion_labels), evasion_labels.view(-1))
            loss = loss_clarity + loss_evasion
        
        return MultiHeadClassifierOutput(
            loss=loss,
            logits_clarity=logits_clarity,
            logits_evasion=logits_evasion,
            hidden_states=None,
            attentions=None,
        )

## Metrics & Trainer

In [ ]:
def compute_metrics(eval_pred):
    """Metrics with standard evaluation (using first/majority label)"""
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(predictions, tuple):
        logits_clarity, logits_evasion = predictions
        if hasattr(logits_clarity, 'cpu'):
            logits_clarity = logits_clarity.cpu().numpy()
        if hasattr(logits_evasion, 'cpu'):
            logits_evasion = logits_evasion.cpu().numpy()
    else:
        logits_clarity = predictions[:, :3]
        logits_evasion = predictions[:, 3:]
        
    preds_clarity = np.argmax(logits_clarity, axis=-1)
    preds_evasion = np.argmax(logits_evasion, axis=-1)
    
    if isinstance(labels, tuple):
        labels_clarity, labels_evasion = labels
        if hasattr(labels_clarity, 'cpu'):
            labels_clarity = labels_clarity.cpu().numpy()
        if hasattr(labels_evasion, 'cpu'):
            labels_evasion = labels_evasion.cpu().numpy()
    else:
        labels_clarity = labels[:, 0] if labels.ndim > 1 else labels
        labels_evasion = labels[:, 1] if labels.ndim > 1 else labels

    accuracy_clarity = accuracy_score(labels_clarity, preds_clarity)
    f1_clarity = f1_score(labels_clarity, preds_clarity, average='macro')

    accuracy_evasion = accuracy_score(labels_evasion, preds_evasion)
    f1_evasion = f1_score(labels_evasion, preds_evasion, average='macro')
    
    return {
        'accuracy_clarity': accuracy_clarity,
        'f1_clarity': f1_clarity,
        'accuracy_evasion': accuracy_evasion,
        'f1_evasion': f1_evasion,
        'f1_combined': (f1_clarity + f1_evasion) / 2,
    }


class MultiHeadTrainer(Trainer):
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss
    
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        clarity_labels = inputs.get('clarity_labels')
        evasion_labels = inputs.get('evasion_labels')
        
        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs.loss
            logits_clarity = outputs.logits_clarity
            logits_evasion = outputs.logits_evasion
        
        if prediction_loss_only:
            return (loss, None, None)
        
        logits = (logits_clarity.detach().float(), logits_evasion.detach().float())

        if clarity_labels is not None and evasion_labels is not None:
            labels = (clarity_labels.detach(), evasion_labels.detach())
        else:
            labels = None
        
        return (loss, logits, labels)


class EarlyStoppingCallback(TrainerCallback):    
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1_combined", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

# 7-Fold Stratified Cross-Validation & Confusion Matrix Analysis

This section performs 7-fold stratified CV on the training set to collect out-of-fold predictions.
We then compute confusion matrices for both Subtask 1 (Clarity) and Subtask 2 (Evasion).

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create output directory for confusion matrices
os.makedirs("./confusion_matrices", exist_ok=True)

print("="*60)
print("7-FOLD STRATIFIED CROSS-VALIDATION")
print("="*60)

# We need to stratify by both clarity and evasion labels
# Create a combined label for stratification
train_df['combined_label'] = train_df['clarity_label'] + "_" + train_df['evasion_label']

n_folds = 7
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

# Storage for out-of-fold predictions
oof_preds_clarity = np.zeros(len(train_df), dtype=int)
oof_preds_evasion = np.zeros(len(train_df), dtype=int)
oof_true_clarity = np.zeros(len(train_df), dtype=int)
oof_true_evasion = np.zeros(len(train_df), dtype=int)

print(f"\nStarting {n_folds}-fold cross-validation...")
print(f"Total training samples: {len(train_df)}\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['combined_label']), 1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}/{n_folds}")
    print(f"{'='*60}")
    print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}")
    
    # Create fold datasets
    fold_train_df = train_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = train_df.iloc[val_idx].reset_index(drop=True)
    
    # Tokenize
    fold_train_dataset = Dataset.from_pandas(
        fold_train_df[['question', 'interview_answer', 'clarity_label', 'evasion_label']], 
        preserve_index=False
    )
    fold_val_dataset = Dataset.from_pandas(
        fold_val_df[['question', 'interview_answer', 'clarity_label', 'evasion_label']], 
        preserve_index=False
    )
    
    fold_train_tokenized = fold_train_dataset.map(tokenize_function, batched=True, remove_columns=fold_train_dataset.column_names)
    fold_val_tokenized = fold_val_dataset.map(tokenize_function, batched=True, remove_columns=fold_val_dataset.column_names)
    
    # Initialize model for this fold
    config = AutoConfig.from_pretrained(MODEL_NAME)
    fold_model = MultiHeadMaxPooling(config)
    
    base_model = AutoModel.from_pretrained(MODEL_NAME)
    fold_model.roberta.load_state_dict(base_model.state_dict())
    del base_model
    
    # Training arguments for this fold
    fold_training_args = TrainingArguments(
        output_dir=f"./results_cv_fold{fold}",
        learning_rate=1e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=15,
        warmup_ratio=0.1,
        weight_decay=0.01,
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        bf16=True,
        bf16_full_eval=True,
        optim="adamw_torch",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1_combined",
        greater_is_better=True,
        logging_steps=50,
        seed=SEED,
        report_to="none",
    )
    
    fold_trainer = MultiHeadTrainer(
        model=fold_model,
        args=fold_training_args,
        train_dataset=fold_train_tokenized,
        eval_dataset=fold_val_tokenized,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1_combined")],
    )
    
    # Train
    print(f"\nTraining fold {fold}...")
    fold_trainer.train()
    
    # Get predictions on validation fold
    print(f"\nGetting predictions for fold {fold}...")
    val_output = fold_trainer.predict(fold_val_tokenized)
    logits_clarity, logits_evasion = val_output.predictions
    
    # Store out-of-fold predictions
    oof_preds_clarity[val_idx] = np.argmax(logits_clarity, axis=-1)
    oof_preds_evasion[val_idx] = np.argmax(logits_evasion, axis=-1)
    oof_true_clarity[val_idx] = [clarity_label2id[label] for label in fold_val_df['clarity_label']]
    oof_true_evasion[val_idx] = [evasion_label2id[label] for label in fold_val_df['evasion_label']]
    
    # Compute fold metrics
    fold_acc_clarity = accuracy_score(oof_true_clarity[val_idx], oof_preds_clarity[val_idx])
    fold_f1_clarity = f1_score(oof_true_clarity[val_idx], oof_preds_clarity[val_idx], average='macro')
    fold_acc_evasion = accuracy_score(oof_true_evasion[val_idx], oof_preds_evasion[val_idx])
    fold_f1_evasion = f1_score(oof_true_evasion[val_idx], oof_preds_evasion[val_idx], average='macro')
    
    print(f"\nFold {fold} Results:")
    print(f"  Clarity - Acc: {fold_acc_clarity:.4f}, F1: {fold_f1_clarity:.4f}")
    print(f"  Evasion - Acc: {fold_acc_evasion:.4f}, F1: {fold_f1_evasion:.4f}")
    
    # Clean up
    del fold_model, fold_trainer
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("CROSS-VALIDATION COMPLETE")
print(f"{'='*60}")

In [ ]:
# Overall CV metrics
overall_acc_clarity = accuracy_score(oof_true_clarity, oof_preds_clarity)
overall_f1_clarity = f1_score(oof_true_clarity, oof_preds_clarity, average='macro')
overall_acc_evasion = accuracy_score(oof_true_evasion, oof_preds_evasion)
overall_f1_evasion = f1_score(oof_true_evasion, oof_preds_evasion, average='macro')

print("\n" + "="*60)
print("OVERALL OUT-OF-FOLD RESULTS")
print("="*60)
print(f"\nClarity (Subtask 1):")
print(f"  Accuracy: {overall_acc_clarity:.4f}")
print(f"  Macro F1: {overall_f1_clarity:.4f}")

print(f"\nEvasion (Subtask 2):")
print(f"  Accuracy: {overall_acc_evasion:.4f}")
print(f"  Macro F1: {overall_f1_evasion:.4f}")

print(f"\nCombined F1: {(overall_f1_clarity + overall_f1_evasion) / 2:.4f}")

In [ ]:
# Detailed Classification Reports
print("\n" + "="*60)
print("DETAILED CLASSIFICATION REPORTS (OUT-OF-FOLD)")
print("="*60)

# Convert predictions to label strings
oof_pred_labels_clarity = [clarity_id2label[p] for p in oof_preds_clarity]
oof_true_labels_clarity = [clarity_id2label[t] for t in oof_true_clarity]
oof_pred_labels_evasion = [evasion_id2label[p] for p in oof_preds_evasion]
oof_true_labels_evasion = [evasion_id2label[t] for t in oof_true_evasion]

print("\n--- CLARITY (Subtask 1) ---")
print(classification_report(oof_true_labels_clarity, oof_pred_labels_clarity, labels=clarity_labels, zero_division=0))

print("\n--- EVASION (Subtask 2) ---")
print(classification_report(oof_true_labels_evasion, oof_pred_labels_evasion, labels=evasion_classes, zero_division=0))

## Confusion Matrices

In [ ]:
def plot_confusion_matrix(cm, labels, title, filename, normalize=False):
    """Plot and save confusion matrix"""
    if normalize:
        cm_display = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.3f'
        cmap = 'Blues'
    else:
        cm_display = cm
        fmt = 'd'
        cmap = 'Blues'
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_display, annot=True, fmt=fmt, cmap=cmap, 
                xticklabels=labels, yticklabels=labels,
                cbar_kws={'label': 'Normalized Count' if normalize else 'Count'})
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename}")
    plt.show()

def cm_to_latex(cm, labels, title):
    """Convert confusion matrix to LaTeX table format"""
    n = len(labels)
    
    # Start table
    latex = "\\begin{table}[h]\n"
    latex += "\\centering\n"
    latex += f"\\caption{{{title}}}\n"
    latex += "\\begin{tabular}{l|" + "c" * n + "}\n"
    latex += "\\hline\n"
    
    # Header row
    latex += " & " + " & ".join([f"\\textbf{{{label}}}" for label in labels]) + " \\\\\n"
    latex += "\\hline\n"
    
    # Data rows
    for i, label in enumerate(labels):
        row = f"\\textbf{{{label}}}"
        for j in range(n):
            row += f" & {cm[i, j]}"
        row += " \\\\\n"
        latex += row
    
    latex += "\\hline\n"
    latex += "\\end{tabular}\n"
    latex += "\\end{table}\n"
    
    return latex

# Label strings already converted in previous cell
print("Helper functions defined for confusion matrix visualization and LaTeX export")

In [ ]:
# SUBTASK 1: Clarity Confusion Matrix (from out-of-fold predictions)
print("\n" + "="*60)
print("SUBTASK 1: CLARITY CONFUSION MATRIX (OUT-OF-FOLD)")
print("="*60)

cm_clarity = confusion_matrix(oof_true_labels_clarity, oof_pred_labels_clarity, labels=clarity_labels)

print("\n--- Raw Confusion Matrix ---")
print(cm_clarity)

# Plot raw CM
plot_confusion_matrix(
    cm_clarity, 
    clarity_labels,
    "Clarity Confusion Matrix (Raw Counts)",
    "./confusion_matrices/clarity_raw.png",
    normalize=False
)

# Plot normalized CM
plot_confusion_matrix(
    cm_clarity, 
    clarity_labels,
    "Clarity Confusion Matrix (Normalized)",
    "./confusion_matrices/clarity_normalized.png",
    normalize=True
)

# LaTeX table
print("\n--- LaTeX Table (Raw) ---")
latex_clarity = cm_to_latex(cm_clarity, clarity_labels, "Clarity Confusion Matrix")
print(latex_clarity)

In [ ]:
# SUBTASK 2: Evasion Confusion Matrix (from out-of-fold predictions)
print("\n" + "="*60)
print("SUBTASK 2: EVASION CONFUSION MATRIX (OUT-OF-FOLD)")
print("="*60)

cm_evasion = confusion_matrix(oof_true_labels_evasion, oof_pred_labels_evasion, labels=evasion_classes)

print("\n--- Raw Confusion Matrix ---")
print(cm_evasion)

# Plot raw CM
plot_confusion_matrix(
    cm_evasion, 
    evasion_classes,
    "Evasion Confusion Matrix (Raw Counts)",
    "./confusion_matrices/evasion_raw.png",
    normalize=False
)

# Plot normalized CM
plot_confusion_matrix(
    cm_evasion, 
    evasion_classes,
    "Evasion Confusion Matrix (Normalized)",
    "./confusion_matrices/evasion_normalized.png",
    normalize=True
)

# LaTeX table
print("\n--- LaTeX Table (Raw) ---")
latex_evasion = cm_to_latex(cm_evasion, evasion_classes, "Evasion Confusion Matrix")
print(latex_evasion)

In [ ]:
# Save normalized confusion matrices as LaTeX tables as well
def cm_to_latex_normalized(cm, labels, title):
    """Convert normalized confusion matrix to LaTeX table format"""
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    n = len(labels)
    
    latex = "\\begin{table}[h]\n"
    latex += "\\centering\n"
    latex += f"\\caption{{{title}}}\n"
    latex += "\\begin{tabular}{l|" + "c" * n + "}\n"
    latex += "\\hline\n"
    
    # Header row
    latex += " & " + " & ".join([f"\\textbf{{{label}}}" for label in labels]) + " \\\\\n"
    latex += "\\hline\n"
    
    # Data rows
    for i, label in enumerate(labels):
        row = f"\\textbf{{{label}}}"
        for j in range(n):
            row += f" & {cm_norm[i, j]:.3f}"
        row += " \\\\\n"
        latex += row
    
    latex += "\\hline\n"
    latex += "\\end{tabular}\n"
    latex += "\\end{table}\n"
    
    return latex

print("\n" + "="*60)
print("NORMALIZED CONFUSION MATRICES (LaTeX)")
print("="*60)

print("\n--- Clarity (Normalized) ---")
latex_clarity_norm = cm_to_latex_normalized(cm_clarity, clarity_labels, "Clarity Confusion Matrix (Normalized)")
print(latex_clarity_norm)

print("\n--- Evasion (Normalized) ---")
latex_evasion_norm = cm_to_latex_normalized(cm_evasion, evasion_classes, "Evasion Confusion Matrix (Normalized)")
print(latex_evasion_norm)

print("\n" + "="*60)
print("All confusion matrices saved to ./confusion_matrices/")
print("="*60)

## Per-Fold Statistics (Mean ± Std)

In [ ]:
# Recreate the same fold splits to compute per-fold statistics
print("="*60)
print("PER-FOLD PERFORMANCE STATISTICS")
print("="*60)

n_folds = 7
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

fold_metrics = {
    'clarity_acc': [],
    'clarity_f1': [],
    'evasion_acc': [],
    'evasion_f1': [],
    'combined_f1': []
}

print(f"\nComputing metrics for each of {n_folds} folds...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['combined_label']), 1):
    # Extract predictions and true labels for this fold
    fold_pred_clarity = oof_preds_clarity[val_idx]
    fold_true_clarity = oof_true_clarity[val_idx]
    fold_pred_evasion = oof_preds_evasion[val_idx]
    fold_true_evasion = oof_true_evasion[val_idx]
    
    # Compute metrics
    acc_clarity = accuracy_score(fold_true_clarity, fold_pred_clarity)
    f1_clarity = f1_score(fold_true_clarity, fold_pred_clarity, average='macro')
    acc_evasion = accuracy_score(fold_true_evasion, fold_pred_evasion)
    f1_evasion = f1_score(fold_true_evasion, fold_pred_evasion, average='macro')
    combined_f1 = (f1_clarity + f1_evasion) / 2
    
    # Store metrics
    fold_metrics['clarity_acc'].append(acc_clarity)
    fold_metrics['clarity_f1'].append(f1_clarity)
    fold_metrics['evasion_acc'].append(acc_evasion)
    fold_metrics['evasion_f1'].append(f1_evasion)
    fold_metrics['combined_f1'].append(combined_f1)
    
    print(f"Fold {fold}:")
    print(f"  Clarity  - Acc: {acc_clarity:.4f}, F1: {f1_clarity:.4f}")
    print(f"  Evasion  - Acc: {acc_evasion:.4f}, F1: {f1_evasion:.4f}")
    print(f"  Combined - F1: {combined_f1:.4f}")
    print()

# Compute mean and std
print("="*60)
print("SUMMARY: MEAN ± STD ACROSS FOLDS")
print("="*60)

print(f"\nClarity (Subtask 1):")
print(f"  Accuracy: {np.mean(fold_metrics['clarity_acc']):.4f} ± {np.std(fold_metrics['clarity_acc']):.4f}")
print(f"  Macro F1: {np.mean(fold_metrics['clarity_f1']):.4f} ± {np.std(fold_metrics['clarity_f1']):.4f}")

print(f"\nEvasion (Subtask 2):")
print(f"  Accuracy: {np.mean(fold_metrics['evasion_acc']):.4f} ± {np.std(fold_metrics['evasion_acc']):.4f}")
print(f"  Macro F1: {np.mean(fold_metrics['evasion_f1']):.4f} ± {np.std(fold_metrics['evasion_f1']):.4f}")

print(f"\nCombined:")
print(f"  F1 Score: {np.mean(fold_metrics['combined_f1']):.4f} ± {np.std(fold_metrics['combined_f1']):.4f}")

# Create a summary table
print("\n" + "="*60)
print("FOLD-WISE RESULTS TABLE")
print("="*60)
print(f"\n{'Fold':<6} {'Clarity Acc':>12} {'Clarity F1':>12} {'Evasion Acc':>12} {'Evasion F1':>12} {'Combined F1':>12}")
print("-" * 78)
for i in range(n_folds):
    print(f"{i+1:<6} {fold_metrics['clarity_acc'][i]:>12.4f} {fold_metrics['clarity_f1'][i]:>12.4f} "
          f"{fold_metrics['evasion_acc'][i]:>12.4f} {fold_metrics['evasion_f1'][i]:>12.4f} "
          f"{fold_metrics['combined_f1'][i]:>12.4f}")
print("-" * 78)
print(f"{'Mean':<6} {np.mean(fold_metrics['clarity_acc']):>12.4f} "
      f"{np.mean(fold_metrics['clarity_f1']):>12.4f} "
      f"{np.mean(fold_metrics['evasion_acc']):>12.4f} "
      f"{np.mean(fold_metrics['evasion_f1']):>12.4f} "
      f"{np.mean(fold_metrics['combined_f1']):>12.4f}")
print(f"{'Std':<6} {np.std(fold_metrics['clarity_acc']):>12.4f} "
      f"{np.std(fold_metrics['clarity_f1']):>12.4f} "
      f"{np.std(fold_metrics['evasion_acc']):>12.4f} "
      f"{np.std(fold_metrics['evasion_f1']):>12.4f} "
      f"{np.std(fold_metrics['combined_f1']):>12.4f}")

In [ ]:
# LaTeX table for paper
print("\n" + "="*60)
print("LATEX TABLE FOR PAPER")
print("="*60)

latex_results = """
\\begin{table}[h]
\\centering
\\caption{7-Fold Cross-Validation Results - Max Pooling Ablation}
\\begin{tabular}{lcccc}
\\hline
\\textbf{Metric} & \\textbf{Clarity Acc} & \\textbf{Clarity F1} & \\textbf{Evasion Acc} & \\textbf{Evasion F1} \\\\
\\hline
"""

for i in range(n_folds):
    latex_results += f"Fold {i+1} & {fold_metrics['clarity_acc'][i]:.4f} & {fold_metrics['clarity_f1'][i]:.4f} & "
    latex_results += f"{fold_metrics['evasion_acc'][i]:.4f} & {fold_metrics['evasion_f1'][i]:.4f} \\\\\n"

latex_results += "\\hline\n"
latex_results += f"Mean & {np.mean(fold_metrics['clarity_acc']):.4f} & {np.mean(fold_metrics['clarity_f1']):.4f} & "
latex_results += f"{np.mean(fold_metrics['evasion_acc']):.4f} & {np.mean(fold_metrics['evasion_f1']):.4f} \\\\\n"
latex_results += f"Std & {np.std(fold_metrics['clarity_acc']):.4f} & {np.std(fold_metrics['clarity_f1']):.4f} & "
latex_results += f"{np.std(fold_metrics['evasion_acc']):.4f} & {np.std(fold_metrics['evasion_f1']):.4f} \\\\\n"
latex_results += """\\hline
\\end{tabular}
\\end{table}
"""

print(latex_results)

## Additional Analysis for Final Report

**What's included in this notebook:**
- ✅ 7-fold stratified cross-validation with mean ± std
- ✅ Overall OOF accuracy and macro F1 scores
- ✅ Detailed per-class classification reports
- ✅ Confusion matrices (raw and normalized) with visualizations
- ✅ LaTeX tables for all results
- ✅ Per-fold performance breakdown

**Consider adding for a complete experiment report:**
1. **Hyperparameters summary** - Document all training settings
2. **Training time** - Total time and per-fold average
3. **Model size** - Number of parameters, memory usage
4. **Comparison baseline** - If you have other pooling methods (mean, attention-based)
5. **Error analysis** - Specific examples where the model fails
6. **Class distribution impact** - Performance vs. class frequency
7. **Confidence calibration** - Check if model is well-calibrated